# 02 — Qwen3.5-9B validation gate (Kaggle T4, 4-bit)

**This cell block downloads the model** (`from_pretrained` = downloader, cached to `/kaggle/working/hf_cache`). T4 rules: `fp16` (no bf16 on Turing), `sdpa` (no flash-attn-2), 4-bit, batch=1, `max_pixels` capped — the cap *is* the visual-budget knob. Fairness: SAME question string (`+ " Output the answer directly."`) both arms; only representation changes.

In [ ]:
# Cell 1 — config (one-line swap on OOM)
MODEL_ID = "Qwen/Qwen3.5-9B"   # only model for this arm — no fallback
BUDGETS = [{"tag": "low", "max_pixels": 512*28*28, "canvas_w": 600}, {"tag": "high", "max_pixels": 1024*28*28, "canvas_w": 1100}]
CONTEXTS = [{"tag": "short", "words": 350}, {"tag": "med", "words": 1500}, {"tag": "long", "words": 3000}]
GEN = {"temperature": 0.0, "max_new_tokens": 128}
OUT = "/kaggle/working/results/raw/qwen35_val.jsonl"
print(MODEL_ID, BUDGETS, GEN)

In [ ]:
# Cell 2 — loader = DOWNLOADER (first from_pretrained pulls weights from HF Hub)
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from transformers import TorchAoConfig

quant = TorchAoConfig("int4_weight_only", group_size=128)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True,
    min_pixels=256*28*28, max_pixels=1024*28*28)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map="auto",
    attn_implementation="sdpa", quantization_config=quant,
    trust_remote_code=True, cache_dir="/tmp/hf_cache")  # weights to /tmp, not working dir
model.eval()
print("loaded:", MODEL_ID, "| device:", model.device if hasattr(model, 'device') else "auto")

In [ ]:
# Cell 3 — renderer (PIL standalone; same logic as 01) + sample generator + metrics
import json, os, random, re, string, textwrap, time, unicodedata
from PIL import Image, ImageDraw, ImageFont

def normalize(t):
    t = unicodedata.normalize("NFKC", t or "")
    return re.sub(r"[ \t]+", " ", t.replace("\r\n", "\n")).strip()

def render(text, width=800, font_size=18, ls=6, wrap=80, pad=30):
    text = normalize(text)
    try: font = ImageFont.truetype("arial.ttf", font_size)
    except IOError: font = ImageFont.load_default()
    lines = []
    for p in text.split("\n\n"):
        lines.extend(textwrap.wrap(p, width=wrap)); lines.append("")
    img = Image.new("RGB", (width, 2*pad + max(len(lines)*(font_size+ls), 100)), "white")
    d = ImageDraw.Draw(img); y = pad
    for ln in lines:
        if ln: d.text((pad, y), ln, fill="black", font=font)
        y += font_size + ls
    return img

DIST = ("The thermal radiator on the secondary boom radiates excess heat into cold space. "
        "Star tracking sensor arrays refresh coordinate quaternions at thirty hertz. ")
def make_sample(words, seed):
    rng = random.Random(seed)
    code = f"CIPHER-{rng.randint(1000, 9999)}-OMEGA"
    needle = f"Special authorization key {code} was recorded in emergency protocol ledger section 9."
    paras, n = [], 0
    while n < words:
        p = " ".join(rng.choice([DIST]) for _ in range(6)); paras.append(p); n += len(p.split())
    w = paras[len(paras)//2].split(); w.insert(len(w)//2, needle)
    paras[len(paras)//2] = " ".join(w)
    return "\n\n".join(paras), code

def norm(s):
    return re.sub(r"\s+", " ", (s or "").lower().translate(str.maketrans("", "", string.punctuation))).strip()
def tok_f1(pred, gt):
    p, g = set(norm(pred).split()), set(norm(gt).split())
    if not p or not g: return 0.0
    tp = len(p & g)
    prec, rec = tp/len(p), tp/len(g)
    return 0.0 if prec+rec == 0 else 2*prec*rec/(prec+rec)

In [ ]:
# Cell 4 — A/B loop: text arm vs image arm, m/q/k accounting, JSONL log (Sec 21)
import torch
from datetime import datetime, timezone
os.makedirs(os.path.dirname(OUT), exist_ok=True)
tok = processor.tokenizer
QSUFFIX = " Output the answer directly."
n = 0
for ci, c in enumerate(CONTEXTS):
    context, gt = make_sample(c["words"], seed=100+ci)
    question = "What special authorization key was recorded in emergency protocol ledger section 9?" + QSUFFIX
    m = len(tok(context, add_special_tokens=False).input_ids)
    q = len(tok(question, add_special_tokens=False).input_ids)
    # --- Arm A: text ---
    msg_t = [{"role": "user", "content": [{"type": "text", "text": f"Context:\n{context}\n\nQuestion: {question}"}]}]
    pt = processor.apply_chat_template(msg_t, tokenize=False, add_generation_prompt=True)
    in_t = processor(text=[pt], padding=True, return_tensors="pt").to(model.device)
    t_text = in_t.input_ids.shape[-1]
    torch.cuda.reset_peak_memory_stats() if torch.cuda.is_available() else None
    s = time.perf_counter()
    with torch.no_grad():
        out_t = model.generate(**in_t, max_new_tokens=GEN["max_new_tokens"], do_sample=False)
    lat_t = time.perf_counter() - s
    pred_t = processor.batch_decode(out_t[:, in_t.input_ids.shape[-1]:], skip_special_tokens=True)[0]
    vram_t = torch.cuda.max_memory_allocated()/1e9 if torch.cuda.is_available() else None
    for b in BUDGETS:
        img = render(context, width=b["canvas_w"])
        msg_i = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": f"Question: {question}"}]}]
        pi = processor.apply_chat_template(msg_i, tokenize=False, add_generation_prompt=True)
        in_i = processor(text=[pi], images=[img], padding=True, return_tensors="pt").to(model.device)
        t_image = in_i.input_ids.shape[-1]          # decoder-visible k+q (+template)
        k_proxy = max(t_image - q, 1)               # visual contribution (labeled proxy)
        torch.cuda.reset_peak_memory_stats() if torch.cuda.is_available() else None
        s = time.perf_counter()
        with torch.no_grad():
            out_i = model.generate(**in_i, max_new_tokens=GEN["max_new_tokens"], do_sample=False)
        lat_i = time.perf_counter() - s
        pred_i = processor.batch_decode(out_i[:, in_i.input_ids.shape[-1]:], skip_special_tokens=True)[0]
        vram_i = torch.cuda.max_memory_allocated()/1e9 if torch.cuda.is_available() else None
        rec = {"experiment_id": "qwen35_val", "timestamp": datetime.now(timezone.utc).isoformat(),
               "model": MODEL_ID, "task": "ruler-sniah", "input_mode": "paired",
               "context_tag": c["tag"], "visual_budget_tag": b["tag"], "max_pixels": b["max_pixels"],
               "image_size": f"{img.width}x{img.height}", "renderer": "pil-fallback",
               "context_text_tokens_m": m, "question_tokens_q": q,
               "decoder_text_arm": t_text, "decoder_image_arm": t_image, "visual_tokens_proxy_k": k_proxy,
               "CR_context_m_over_k": round(m/k_proxy, 3), "CR_decoder_text_over_image": round(t_text/t_image, 3),
               "hit_text": int(norm(gt) in norm(pred_t)), "hit_image": int(norm(gt) in norm(pred_i)),
               "f1_text": round(tok_f1(pred_t, gt), 3), "f1_image": round(tok_f1(pred_i, gt), 3),
               "lat_text_s": round(lat_t, 2), "lat_image_s": round(lat_i, 2),
               "vram_text_gb": vram_t, "vram_image_gb": vram_i, "ground_truth": gt,
               "pred_text": pred_t.strip()[:300], "pred_image": pred_i.strip()[:300]}
        with open(OUT, "a") as f: f.write(json.dumps(rec) + "\n")
        n += 1
        print(f"[{n}/6] {c['tag']} x {b['tag']}: m={m} imgdec={t_image} k~{k_proxy} CRctx={rec['CR_context_m_over_k']} hitT/I={rec['hit_text']}/{rec['hit_image']} latT/I={lat_t:.1f}/{lat_i:.1f}s")
print(f"\nLOGGED {n} paired rows -> {OUT}")

In [ ]:
# Cell 5 — aggregate from the JSONL (never hand-edit): accuracy + mean F1 + mean CR per budget
import pandas as pd
df = pd.read_json(OUT, lines=True)
g = df.groupby("visual_budget_tag").agg(n=("hit_image", "size"), acc_text=("hit_text", "mean"),
    acc_image=("hit_image", "mean"), f1_image=("f1_image", "mean"),
    cr_ctx=("CR_context_m_over_k", "mean"), lat_t=("lat_text_s", "mean"), lat_i=("lat_image_s", "mean"))
print(g.round(3).to_string())
print("\nGATE: pass iff all 6 rows logged, images uncropped (see 01), and k/m/CR columns non-null.")